In [0]:
from pyspark.sql.types import *

Schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("Product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("Order_Date", StringType(), True),
    StructField("status", StringType(), True)
])

stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .schema(Schema)
    .load("/Volumes/workspace/default/retail_data/streaming_input")
)

In [0]:
query = (
    stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "/Volumes/workspace/default/retail_data/checkpoints/orders_stream")
    .toTable("orders_stream")
)

In [0]:
spark.sql("drop table orders_stream")

In [0]:
dbutils.fs.rm("/Volumes/workspace/default/retail_data/checkpoints/orders_stream", True)

In [0]:
spark.sql("select count(*) from orders_stream limit 10").show()

In [0]:
spark.sql("select * from orders_stream").show(truncate=False)